# 📝 과제 02: 택배 회사 배송 문의 모델의 장비와 양자화 형식 정하기

## 만들 결과

택배 회사가 배송 문의에 답할 모델을 사내 GPU에 올리려 합니다. 교안 02의 메모리 계산, 정밀도, 양자화 오차, 양자화 형식 읽기를 적용해 장비와 형식을 정할 근거를 만듭니다.

<img src="images/과제/41_gwaje02_flow.png" width="1050" alt="제목은 '배송 문의 모델의 장비와 양자화 형식 정하기'입니다. 카드 여섯 장이 두 줄로 이어집니다. 1 GPU 고르기(16 · 24 · 48 · 80GB), 2 정밀도(inf · 0), 3 층마다 이상치(최대 / 99% 분위), 4 묶음 크기(오차 · 비트), 5 GGUF 형식 고르기(비트 · 토큰/초), 6 AWQ · GPTQ 설정(비트 · 대칭). 카드 옆에 노트북을 든 개발자가 있습니다.">

- 모델 세 개와 정밀도 세 개마다 들어가는 가장 작은 GPU를 고릅니다(Qwen3.5-35B-A3B BF16은 80GB 한 장도 넘음).
- 네 값을 BF16과 FP16으로 바꿔 범위를 넘거나 0이 되는 값을 찾습니다.
- Qwen3.5-4B 층 다섯 개의 이상치를 비교해 양자화가 가장 어려운 층을 찾습니다.
- 그 층을 4비트로 묶음 크기를 바꿔 양자화해 오차와 저장 비트를 비교합니다.
- 커뮤니티가 Hub에 올린 Qwen3.5-4B GGUF 파일의 크기로 형식마다 가중치당 비트를 계산해 4GB GPU에 올릴 파일을 고르고, Ollama 8비트·4비트 태그의 토큰/초를 비교합니다.
- 커뮤니티가 Qwen3.5-4B를 양자화해 올린 AWQ·GPTQ 저장소의 설정에서 비트, 묶음, 대칭 여부를 비교합니다.

## 준비: 모델과 제공 코드

- **실행 모델:** 교안 02와 같은 `Qwen/Qwen3.5-4B`(Transformers, 3·4번에서 층 가중치를 꺼냄)와 Ollama `qwen3.5:4b-q8_0`·`qwen3.5:4b-q4_K_M`(5번)입니다. Ollama 서버가 켜져 있어야 합니다. 6번은 Hub의 설정 파일만 조회합니다.
- **맨 위 셀:** 실습 모델 파일을 미리 내려받습니다. 커널을 다시 시작할 때도 이 셀부터 실행합니다(받은 파일은 확인만 합니다).
- **시작 전:** 앞 노트북의 커널을 종료(Shut Down)합니다. 준비 셀이 Ollama가 GPU에 올려 둔 모델을 내립니다.
- **제공 코드:** 준비 셀(정밀도별 바이트와 GPU 크기 포함), 교안 02의 `model_memory_gib`, `quantize_symmetric`, `measure_tag`, 모델 불러오기
- **작성:** `[작성]` 셀의 `...`를 채우고 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import asyncio

from huggingface_hub import snapshot_download
import ollama

# 실습에서 쓸 모델을 미리 내려받습니다. 처음 한 번만 오래 걸리고, 다음부터는 이미 받은 파일을 확인만 합니다
# Transformers용 원본 모델(BF16, 약 9GB): Hugging Face 캐시 폴더(기본 ~/.cache/huggingface/hub)에 저장됩니다
# snapshot_download는 일반 함수라서 asyncio.to_thread로 감싸야 다른 다운로드와 함께 기다릴 수 있습니다
tasks = [asyncio.to_thread(snapshot_download, "Qwen/Qwen3.5-4B")]

# Ollama용 모델(GGUF 파일): pull 요청을 받은 Ollama 서버가 내려받아 서버의 모델 폴더(기본 ~/.ollama/models)에 보관합니다
# 태그 이름은 ollama.com/search 에서 모델을 검색하고 모델 페이지의 Tags 탭에서 고릅니다
ollama_client = ollama.AsyncClient()
ollama_tags = ["qwen3.5:4b-q8_0", "qwen3.5:4b-q4_K_M"]
for tag in ollama_tags:
    tasks.append(ollama_client.pull(tag))

# 두 다운로드는 받는 곳과 저장 위치가 달라 서로 기다릴 필요가 없으므로 동시에 받고, 결과는 tasks 순서대로 받습니다
results = await asyncio.gather(*tasks)
model_dir = results[0]
print("Hugging Face 모델:", model_dir)
print("Ollama 모델:", ollama_tags)

In [ ]:
import math
import statistics

import pandas as pd

import torch

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다 (지난 단원과 같은 코드)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

from huggingface_hub import HfApi
from transformers import AutoConfig

# 정밀도마다 파라미터 하나의 바이트(교안 02 2-3)와 검토할 GPU 메모리 사양(GB, GiB로 읽음)
precision_bytes = {"BF16": 2, "INT8": 1, "INT4": 0.5}
gpu_sizes = [16, 24, 48, 80]

# 테이블에서 긴 값이 잘리지 않게 합니다
pd.set_option("display.max_colwidth", None)

In [ ]:
# 교안 02에서 만든 메모리 계산, 대칭 양자화, Ollama 태그 측정 함수입니다
def model_memory_gib(n_params, bytes_per_param):
    """파라미터 수 * 파라미터 하나의 바이트를 GiB로 바꿔 돌려줍니다."""
    return n_params * bytes_per_param / 1024 ** 3


def quantize_symmetric(weights, bits, group_size):
    """weights를 bits비트로 대칭 양자화했다가 복원한 값을 원래 모양으로 돌려줍니다. group_size가 0이면 스케일을 하나만 둡니다."""
    # bits비트 정수로 담을 가장 큰 값: INT8은 127, INT4는 7
    max_int = 2 ** (bits - 1) - 1

    # 스케일을 함께 쓸 묶음으로 나눕니다. 한 줄이 한 묶음입니다
    if group_size == 0:
        groups = weights.reshape(1, -1)
    else:
        groups = weights.reshape(-1, group_size)

    # 묶음마다 최대 절댓값 / max_int 로 스케일을 정하고, 나누고 반올림해 정수로 만듭니다
    group_scale = groups.abs().amax(dim=1, keepdim=True) / max_int
    group_q = torch.round(groups / group_scale)

    # 정수에 스케일을 다시 곱해 복원하고 원래 모양으로 되돌립니다
    restored_groups = group_q * group_scale
    return restored_groups.reshape(weights.shape)


def measure_tag(tag, messages):
    """tag로 한 번 미리 생성한 뒤 3번 생성해 토큰/초 중앙값과 마지막 답을 돌려주고, 모델을 GPU에서 내립니다."""
    # 같은 답이 나오도록 temperature 0, seed 0으로 두고 최대 150토큰까지 생성합니다
    options = {"temperature": 0, "seed": 0, "num_predict": 150}

    # 첫 호출은 모델을 GPU에 올리고 준비하는 일이 겹쳐 생성 속도가 흔들리므로 결과를 버립니다
    ollama.chat(model=tag, messages=messages, think=False, options=options)

    # 3번 생성해 토큰/초를 모읍니다. eval_duration은 나노초 단위라 1e9를 곱해 초 단위로 맞춥니다
    speeds = []
    for _ in range(3):
        response = ollama.chat(model=tag, messages=messages, think=False, options=options)
        speeds.append(response.eval_count / response.eval_duration * 1e9)

    # 다음 태그 측정에 이 모델이 섞이지 않게 GPU에서 내립니다
    ollama.generate(model=tag, keep_alive=0)
    return statistics.median(speeds), response.message.content

In [ ]:
# 앞에서 Ollama가 GPU에 올려 둔 모델이 있으면 내립니다. keep_alive=0은 "지금 바로 GPU에서 내리기"라는 뜻입니다
for loaded in ollama.ps().models:
    ollama.generate(model=loaded.model, keep_alive=0)
print("Ollama가 GPU에 올린 모델:", ollama.ps().models)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

# 토크나이저를 불러옵니다. 파일은 맨 위 미리 내려받기 셀이 받아 두었으므로 캐시에서 읽습니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Qwen3.5-4B는 이미지도 받는 모델이지만, AutoModelForCausalLM은 글 생성 부분만 불러옵니다
# 모델 파일에 저장된 정밀도(BF16) 그대로 계산 장치에 올립니다(GPU 메모리 약 8GB). 불러온 모델은 이미 평가 모드입니다
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16).to(device)
print("불러온 모델:", MODEL_NAME, "| 정밀도:", model.dtype, "| 장치:", model.device)

## 1. 가장 작은 GPU 고르기

**배경**: 택배 회사가 살 수 있는 GPU는 16·24·48·80GB입니다. 같은 모델도 정밀도에 따라 들어가는 GPU가 달라지므로, 모델과 정밀도마다 들어가는 가장 작은 GPU를 테이블로 만듭니다(교안 02 2-4). MoE 모델도 메모리는 전체 파라미터로 계산합니다(교안 01 2-3).

**요구사항**
- 제공한 `delivery_models`의 모델 3개 × `precision_bytes`의 정밀도 3개, 모두 9행인 DataFrame **`gpu_table`** 을 만듭니다. 행은 `delivery_models` 순서대로, 모델마다 BF16, INT8, INT4 순서로 쌓습니다. 열은 아래 네 개입니다.
  - **`모델`**, **`정밀도`**: 모델 이름과 정밀도 이름
  - **`생성 시(GiB)`**: `model_memory_gib`로 계산한 가중치 메모리에 20%를 더한 값
  - **`GPU`**: `gpu_sizes`를 작은 것부터 보며 `생성 시(GiB)`가 그 크기 이하인 첫 GPU를 `"16GB"`처럼 숫자 뒤에 GB를 붙인 문자열로, 80GB에도 들어가지 않으면 `"여러 장"`

**예시**: 생성 시 20 GiB면 16GB에는 안 들어가고 24GB에 들어가므로 `"24GB"`입니다.

**확인 기준**: 9행, 네 열 이름, 행마다 `생성 시(GiB)`가 기대값과 0.05 이내이고 `GPU` 문자열이 맞습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 모델과 정밀도를 두 겹 for문으로 돌며 행(dict)을 쌓습니다. GPU는 안쪽에서 gpu_sizes를 한 번 더 돌며 처음 들어가는 크기를 찾습니다.

세부구현:
1. GPU 칸의 기본값을 "여러 장"으로 두고, 들어가는 크기를 찾으면 그 문자열로 바꾼 뒤 반복을 멈춥니다(break).
2. 크기 문자열은 숫자를 문자열로 바꿔 "GB"를 붙입니다.
```

</details>

In [ ]:
# 배송 문의 모델 후보: (이름, 파라미터 수). Qwen3.5-4B는 AutoModelForCausalLM이 불러오는 글 생성 부분의 개수,
# 나머지 두 모델은 Hub의 safetensors 기준 전체 개수입니다
delivery_models = [("Qwen3.5-4B(글 부분)", 4_205_751_296), ("Qwen3-14B", 14_768_307_200), ("Qwen3.5-35B-A3B", 35_951_822_704)]

In [ ]:
# [작성] 모델과 정밀도마다 들어가는 가장 작은 GPU를 테이블로 만드세요.
gpu_rows = []
...

gpu_table = pd.DataFrame(gpu_rows)
display(gpu_table.round(2))

In [ ]:
# [자가채점] 행 수, 열 이름, 메모리와 GPU 판정을 확인합니다.
assert isinstance(gpu_table, pd.DataFrame) and len(gpu_table) == 9, "gpu_table은 모델 3개와 정밀도 3개의 조합, 9행입니다."
assert list(gpu_table.columns) == ["모델", "정밀도", "생성 시(GiB)", "GPU"], "열은 모델, 정밀도, 생성 시(GiB), GPU 순서입니다."
expected_gib = [9.40, 4.70, 2.35, 33.01, 16.50, 8.25, 80.36, 40.18, 20.09]
for got, want, (_, row) in zip(gpu_table["생성 시(GiB)"], expected_gib, gpu_table.iterrows()):
    assert abs(got - want) < 0.05, f"{row['모델']} {row['정밀도']}의 생성 시 메모리가 {got:.2f} GiB입니다. model_memory_gib 결과에 1.2를 곱했는지 확인하세요."
expected_gpu = ["16GB", "16GB", "16GB", "48GB", "24GB", "16GB", "여러 장", "48GB", "24GB"]
for got, want, (_, row) in zip(gpu_table["GPU"], expected_gpu, gpu_table.iterrows()):
    assert got == want, f"{row['모델']} {row['정밀도']}의 GPU가 {got}입니다. 작은 GPU부터 보며 처음 들어가는 크기를 '24GB'처럼 쓰고, 80GB도 넘으면 '여러 장'입니다."
print("✅ 1번 통과")

**결과 읽기**
- **Qwen3.5-35B-A3B BF16 `여러 장`:** 생성 시 80.36 GiB라 80GB 한 장을 조금 넘습니다. INT8이면 48GB, INT4면 24GB 한 장에 들어갑니다.
- **Qwen3-14B BF16 `48GB`:** 생성 시 33.01 GiB라 32GB에도 1 GiB 차이로 들어가지 않습니다. 8비트로 줄이면 24GB, 4비트면 16GB 한 장에 들어갑니다.
- **Qwen3.5-4B BF16 `16GB`:** 글 생성 부분만 불러오면 생성 시 9.40 GiB라 BF16 그대로 16GB 한 장에 들어갑니다. 이미지 처리 부분까지 담은 전체 4,659,865,088개로 계산해도 약 10.4 GiB라 결과는 같습니다.
- **같은 GPU 안에서:** 24GB 한 장이면 4B는 BF16 그대로, 14B는 INT8로, 35B-A3B는 INT4로 줄여야 올릴 수 있습니다.

## 2. 세 정밀도로 바꿔 보기

**배경**: FP16은 담을 수 있는 범위가 좁아, 아주 큰 값은 무한대(`inf`)가 되고 아주 작은 값은 0이 됩니다. BF16은 범위가 넓은 대신 자릿수가 적습니다(교안 02 2-1·2-2).

**요구사항**
- (1) 제공한 `test_values`를 FP32 텐서로 만든 뒤 BF16, FP16으로 바꿔, 세 열 **`원래 값`**, **`BF16`**, **`FP16`**(파이썬 실수)을 가진 DataFrame **`precision_check`** 를 만듭니다.
- (2) FP16으로 바꾼 값이 `inf`인 원래 값들을 리스트 **`fp16_inf`** 에, 0이 된 원래 값들을 리스트 **`fp16_zero`** 에 담습니다. 원래 값은 `test_values`의 값을 그대로 담습니다.

**확인 기준**: 4행, 세 열 이름이고, `fp16_inf`와 `fp16_zero`가 각각 값 하나씩이며 기대하는 원래 값과 같습니다. BF16 열에는 `inf`도 0도 없습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 2-2처럼 텐서를 만들고 .to()로 정밀도를 바꾼 뒤, 테이블에 넣으려고 파이썬 숫자 목록으로 꺼냅니다.

세부구현:
1. 무한대인지는 math 모듈의 함수로 확인합니다.
2. 원래 값과 FP16 값을 zip으로 함께 돌며 조건에 맞는 원래 값을 리스트에 담습니다.
```

</details>

In [ ]:
# [작성] (1) 네 값을 BF16, FP16으로 바꿔 테이블로 만드세요.
test_values = [0.5, 1e-8, 300000.0, 2.718281828]

precision_check = ...
display(precision_check)

In [ ]:
# [작성] (2) FP16에서 inf가 된 값과 0이 된 값을 찾으세요.
fp16_inf = ...
fp16_zero = ...

print("FP16에서 inf:", fp16_inf, "| FP16에서 0:", fp16_zero)

In [ ]:
# [자가채점] 테이블의 모양과 FP16에서 범위를 벗어난 값을 확인합니다.
assert isinstance(precision_check, pd.DataFrame) and len(precision_check) == 4, "precision_check는 test_values 네 개의 4행 DataFrame입니다."
assert list(precision_check.columns) == ["원래 값", "BF16", "FP16"], "열은 원래 값, BF16, FP16 순서입니다."
assert not any(math.isinf(v) for v in precision_check["BF16"]) and all(v != 0 for v in precision_check["BF16"]), "BF16 열에는 inf도 0도 없어야 합니다. BF16으로 바꿨는지 확인하세요."
assert isinstance(fp16_inf, list) and len(fp16_inf) == 1 and math.isclose(fp16_inf[0], 300000.0, rel_tol=1e-6), "fp16_inf에는 FP16에서 inf가 된 원래 값 하나가 들어 있어야 합니다."
assert isinstance(fp16_zero, list) and len(fp16_zero) == 1 and math.isclose(fp16_zero[0], 1e-8, rel_tol=1e-6), "fp16_zero에는 FP16에서 0이 된 원래 값 하나가 들어 있어야 합니다."
print("✅ 2번 통과")

**결과 읽기**
- **300000 -> FP16 `inf`, BF16 299008:** FP16의 최댓값 65504를 넘어 무한대가 됩니다. BF16은 범위가 넓어 담지만 자릿수가 모자라 299008이 됩니다.
- **1e-8 -> FP16 0:** FP16이 담을 수 있는 가장 작은 양수(약 6e-8, 교안의 `tiny`보다 자릿수를 줄여 담는 값)보다도 작아 0이 됩니다. BF16은 지수 비트가 FP32와 같아 1e-8 근처 값을 담습니다.
- **2.718281828 -> 2.71875:** 이 값에서는 두 16비트 형식이 우연히 같은 값이 됩니다. 일반적으로는 가수 비트가 더 많은 FP16이 원래 값에 더 가깝습니다.

## 3. 층마다 이상치 비교

**배경**: 이상치(outlier)가 큰 층일수록 텐서 하나에 스케일 하나를 두는 양자화의 오차가 커집니다(교안 02 4-1). Qwen3.5-4B의 32개 층 가운데 다섯 개의 `mlp.down_proj` 가중치를 비교해 양자화가 가장 어려운 층을 찾습니다.

**요구사항**
- 층 인덱스 `[2, 9, 16, 23, 30]`마다 `model.model.layers[인덱스].mlp.down_proj` 가중치를 FP32로 꺼내(교안 02처럼 인덱스는 0부터 셉니다. 인덱스 2는 3번째 층), 아래 다섯 열을 가진 DataFrame **`outlier_table`**(5행)을 만듭니다.
  - **`층`**: 층 인덱스(정수)
  - **`99% 분위 절댓값`**, **`최대 절댓값`**: 절댓값의 99% 분위와 최댓값. 99% 분위는 교안 02 4-1처럼 `kthvalue`로 구합니다(가중치가 약 2,359만 개라 `torch.quantile`은 쓰지 못합니다).
  - **`최대 / 99% 분위`**: 최대 절댓값 / 99% 분위 절댓값
  - **`INT8 평균 절대 오차`**: `quantize_symmetric`으로 텐서 하나 스케일(`group_size=0`) INT8 양자화·복원했을 때 원래 값과의 평균 절대 오차
- `최대 / 99% 분위`가 가장 큰 층의 인덱스를 정수 **`worst_layer`** 에 담습니다.

**확인 기준**: 5행, 다섯 열 이름이고, 층마다 99% 분위보다 큰 가중치가 약 1%이며, INT8 평균 절대 오차가 0보다 크고 스케일의 절반(최대 절댓값 / 254)을 넘지 않습니다. `worst_layer`가 비율이 가장 큰 층입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 4-1에서 한 층으로 계산한 통계를 층 번호 for문 안에서 반복하고, 행을 dict로 쌓아 테이블로 만듭니다.

세부구현:
1. 가중치는 detach()로 떼어 FP32로 바꾼 뒤 절댓값을 1차원으로 펼칩니다.
2. 99% 분위는 전체 개수의 99% 번째로 작은 값입니다. 그 순번을 정수로 만들어 kthvalue에 넣습니다.
3. 양자화 오차는 제공 함수의 결과와 원래 가중치의 차이에 절댓값을 씌워 평균을 냅니다.
4. 가장 큰 비율의 행을 찾아 그 행의 층 번호를 int로 꺼냅니다.
```

</details>

In [ ]:
# [작성] 층 다섯 개의 이상치 통계와 INT8 오차를 테이블로 만들고, 가장 어려운 층을 찾으세요.
outlier_rows = []
...

outlier_table = pd.DataFrame(outlier_rows)
display(outlier_table)

worst_layer = ...
print("이상치가 가장 큰 층:", worst_layer)

In [ ]:
# [자가채점] 테이블의 모양, 분위와 오차가 성질에 맞는지, 가장 어려운 층을 확인합니다.
assert isinstance(outlier_table, pd.DataFrame) and len(outlier_table) == 5, "outlier_table은 층 다섯 개의 5행 DataFrame입니다."
assert list(outlier_table.columns) == ["층", "99% 분위 절댓값", "최대 절댓값", "최대 / 99% 분위", "INT8 평균 절대 오차"], "열 이름과 순서를 지문과 같게 쓰세요."
assert list(outlier_table["층"]) == [2, 9, 16, 23, 30], "층 열은 2, 9, 16, 23, 30 순서입니다."
for _, row in outlier_table.iterrows():
    layer = int(row["층"])
    check_abs = model.model.layers[layer].mlp.down_proj.weight.detach().float().abs().flatten()

    # 99% 분위라면 그보다 큰 가중치가 전체의 약 1%입니다
    above_share = (check_abs > row["99% 분위 절댓값"]).float().mean().item()
    assert 0.009 < above_share < 0.011, f"{layer}층에서 99% 분위보다 큰 가중치가 {above_share:.2%}입니다. 절댓값으로 0.99 위치의 값을 구했는지 확인하세요."
    assert row["99% 분위 절댓값"] < row["최대 절댓값"] and abs(row["최대 / 99% 분위"] - row["최대 절댓값"] / row["99% 분위 절댓값"]) < 1e-6 * row["최대 / 99% 분위"], f"{layer}층의 최대 / 99% 분위는 최대 절댓값을 99% 분위 절댓값으로 나눈 값입니다."

    # 반올림 오차는 스케일(최대 절댓값 / 127)의 절반을 넘지 않습니다
    assert 0 < row["INT8 평균 절대 오차"] <= row["최대 절댓값"] / 254, f"{layer}층의 INT8 평균 절대 오차가 스케일의 절반보다 큽니다. quantize_symmetric(가중치, 8, 0)으로 INT8 텐서 하나 스케일을 썼는지 확인하세요."

    # 텐서 하나 스케일은 32개 묶음 스케일보다 오차가 커야 합니다
    check_weights = model.model.layers[layer].mlp.down_proj.weight.detach().float()
    group32_error = torch.nn.functional.l1_loss(quantize_symmetric(check_weights, 8, 32), check_weights).item()
    assert row["INT8 평균 절대 오차"] > group32_error, f"{layer}층의 INT8 오차가 32개 묶음 스케일보다 작습니다. group_size=0(텐서 하나)으로 양자화했는지 확인하세요."
assert isinstance(worst_layer, int) and worst_layer == int(outlier_table.loc[outlier_table["최대 / 99% 분위"].idxmax(), "층"]), "worst_layer는 최대 / 99% 분위가 가장 큰 층의 번호(정수)입니다."
print("✅ 3번 통과")

**결과 읽기**
- **`최대 / 99% 분위`:** 가중치 99%가 들어가는 범위보다 최댓값이 몇 배 멀리 떨어져 있는지를 나타냅니다. 이 값이 큰 층일수록 텐서 하나 스케일이 소수의 이상치에 맞춰져, 나머지 가중치가 쓰는 정수 눈금이 적어집니다.
- **INT8 평균 절대 오차:** 반올림 오차는 스케일(최대 절댓값 / 127)의 절반을 넘지 못하므로, 같은 INT8이라도 최대 절댓값이 큰 층에서 오차가 커집니다. 테이블에서 최대 절댓값과 오차를 나란히 비교합니다.
- **`worst_layer`:** 4번에서 이 층으로 묶음 크기를 바꿔 가며 4비트 오차를 비교합니다.

## 4. 묶음 크기 실험

**배경**: 묶음(블록)을 작게 하면 이상치의 영향이 좁아져 오차가 줄지만, 묶음마다 스케일을 하나씩 더 저장해야 합니다(교안 02 3-2, 4-2). 이상치가 가장 큰 `worst_layer`로 4비트의 묶음 크기를 비교합니다.

**요구사항**
- `worst_layer`의 `mlp.down_proj` 가중치를 FP32로 꺼내, group_size `[16, 32, 64, 128, 256]`마다 `quantize_symmetric`으로 INT4(`bits=4`) 양자화·복원한 결과로 DataFrame **`group_table`**(5행)을 만듭니다. 열은 아래 네 개입니다.
  - **`group_size`**
  - **`평균 절대 오차`**: 원래 값과 복원값의 평균 절대 오차
  - **`0이 된 비율`**: 복원값이 정확히 0인 가중치의 비율
  - **`가중치당 비트`**: 4비트 정수 group_size개에 16비트 스케일 하나를 함께 저장할 때 가중치 하나에 드는 비트(교안 02 3-2의 8.5비트 계산과 같은 방식)

**확인 기준**: 5행, 네 열 이름이고, `worst_layer`의 가중치를 썼으며, 묶음이 커질수록 평균 절대 오차와 0이 된 비율이 줄지 않고, 오차가 같은 묶음의 INT8보다 크며(4비트), 가중치당 비트가 정확합니다.

In [ ]:
# [작성] 묶음 크기마다 INT4 오차, 0이 된 비율, 가중치당 비트를 테이블로 만드세요.
worst_weights = ...

group_rows = []
...

group_table = pd.DataFrame(group_rows)
display(group_table)

In [ ]:
# [자가채점] 테이블의 모양, 묶음 크기에 따른 변화, 가중치당 비트를 확인합니다.
assert isinstance(group_table, pd.DataFrame) and len(group_table) == 5, "group_table은 group_size 다섯 개의 5행 DataFrame입니다."
assert list(group_table.columns) == ["group_size", "평균 절대 오차", "0이 된 비율", "가중치당 비트"], "열 이름과 순서를 지문과 같게 쓰세요."
assert list(group_table["group_size"]) == [16, 32, 64, 128, 256], "group_size는 16, 32, 64, 128, 256 순서입니다."
assert torch.equal(worst_weights, model.model.layers[worst_layer].mlp.down_proj.weight.detach().float()), "worst_weights는 3번에서 찾은 worst_layer의 down_proj 가중치(FP32)입니다."
errors = list(group_table["평균 절대 오차"])
zeros = list(group_table["0이 된 비율"])
assert all(a <= b for a, b in zip(errors, errors[1:])), "묶음이 커질수록 평균 절대 오차가 줄었습니다. bits=4와 group_size를 제대로 넘겼는지 확인하세요."
assert all(a <= b for a, b in zip(zeros, zeros[1:])) and 0 < zeros[0] < 1, "0이 된 비율은 복원값이 0인 가중치의 비율이고, 묶음이 커질수록 늘어야 합니다."

# 4비트는 같은 묶음 크기의 8비트보다 눈금이 거칠어 오차가 커야 합니다
int8_group16_error = torch.nn.functional.l1_loss(quantize_symmetric(worst_weights, 8, 16), worst_weights).item()
assert errors[0] > int8_group16_error, "group_size 16의 오차가 같은 묶음의 INT8 오차보다 작습니다. bits=4로 양자화했는지 확인하세요."
assert list(group_table["가중치당 비트"]) == [5.0, 4.5, 4.25, 4.125, 4.0625], "가중치당 비트는 (4비트 정수 group_size개 + 16비트 스케일 하나) / group_size입니다."
print("✅ 4번 통과")

**결과 읽기**
- **group_size 16 -> 256:** 묶음이 커질수록 평균 절대 오차와 0이 된 비율이 늘어납니다. 묶음이 클수록 한 묶음 안에 이상치가 섞일 가능성이 커지고, 그 이상치가 묶음 전체의 눈금을 넓히기 때문입니다.
- **가중치당 비트 5 -> 4.06:** 묶음이 작을수록 스케일을 자주 저장해 비트가 늘어납니다. 4비트 형식이 묶음 32~128을 많이 쓰는 이유가 이 오차와 저장 비트의 맞바꿈입니다(6번의 AWQ·GPTQ는 128).

## 5. GGUF 형식의 비트와 토큰/초

**배경**: 택배 지점 사무실 PC는 GPU 메모리가 4GB라 GGUF 파일로 실행합니다. Qwen은 Qwen3.5-4B의 GGUF 저장소를 직접 올리지 않아, 교안 02 6-2처럼 모델 트리(Quantizations)에서 커뮤니티 저장소를 찾습니다. 그 가운데 LM Studio 팀이 올린 `lmstudio-community/Qwen3.5-4B-GGUF`에는 양자화 형식이 다른 언어 모델 파일 세 개와, 이미지를 처리하는 부분만 담은 `mmproj` 파일이 있습니다(이미지 처리 부분은 교안 02 6-3). 파일 크기로 형식마다 가중치당 비트를 계산하고(교안 02 6-3), 4GB에 들어가는 파일 가운데 비트가 가장 많은 파일을 고릅니다. 이어서 본사 GPU에서 Ollama 8비트·4비트 태그의 토큰/초를 비교합니다(교안 02 1-3, 7절).

**요구사항**
- (1) 교안 02 2-3처럼 `HfApi().model_info(저장소 이름, files_metadata=True)`로 `lmstudio-community/Qwen3.5-4B-GGUF`의 파일 목록을 받아, 이름이 `.gguf`로 끝나고 `mmproj`로 시작하지 않는 파일마다 한 행인 DataFrame **`gguf_files`**(3행)를 만듭니다. 배송 문의는 글만 다루므로 이미지 처리 파일은 뺍니다. 열은 아래 네 개입니다.
  - **`양자화 수준`**: 파일 이름 `Qwen3.5-4B-Q4_K_M.gguf`에서 앞의 `Qwen3.5-4B-`와 뒤의 `.gguf`를 뗀 문자열(예: `"Q4_K_M"`)
  - **`크기(바이트)`**: 파일 크기
  - **`크기(GiB)`**: 파일 크기 / 1024³
  - **`가중치당 비트`**: 파일 크기 × 8 / 제공한 `n_params_4b`
- (2) `크기(GiB)` × 1.2가 4 이하인 행 가운데 `가중치당 비트`가 가장 큰 행의 `양자화 수준`을 문자열 **`office_pick`** 에 담습니다.
- (3) 교안 02의 `measure_tag`로 `tags`의 두 태그에 제공한 `delivery_messages`를 넣어, 태그마다 한 행인 DataFrame **`tag_compare`**(열 `태그`, `토큰/초 중앙값`, `답`)를 만듭니다.
- (4) `gguf_files`에서 Q8_0의 가중치당 비트 / Q4_K_M의 가중치당 비트를 **`bits_ratio`** 에, `tag_compare`에서 q4_K_M 태그의 토큰/초 / q8_0 태그의 토큰/초를 **`speed_ratio`** 에 담습니다.

**확인 기준**: `gguf_files`가 3행, 네 열 이름이고, 형식마다 가중치당 비트가 기대값과 0.01 이내이며, `office_pick`이 4GB 조건을 만족하는 형식 가운데 비트가 가장 많은 형식입니다. `tag_compare`가 두 태그 순서의 2행이고 토큰/초가 0보다 크며 답이 비어 있지 않습니다. `bits_ratio`는 1보다 큽니다(4비트 파일이 더 작음). 토큰/초 값 자체는 장치마다 달라 채점하지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02 2-3의 siblings 반복에서 .gguf로 끝나고 mmproj로 시작하지 않는 파일만 골라 행(dict)을 쌓습니다.

세부구현:
1. 이름 조건은 문자열의 endswith와 startswith로 확인합니다.
2. 양자화 수준은 파일 이름 문자열에서 앞뒤 글자를 빈 문자열로 바꿔 지웁니다.
3. 4GB 조건으로 먼저 행을 거른 뒤, 가중치당 비트로 정렬하거나 가장 큰 값의 행을 찾습니다.
4. 비율은 set_index로 양자화 수준이나 태그를 인덱스로 바꾸면 값을 이름으로 꺼낼 수 있습니다.
```

</details>

In [ ]:
# 불러온 Qwen3.5-4B 글 생성 부분의 파라미터 수(교안 02 2-3과 같은 계산)
n_params_4b = sum(p.numel() for p in model.parameters())
print("파라미터 수:", n_params_4b)

# (3)에서 두 태그에 넣을 배송 문의
delivery_messages = [
    {"role": "system", "content": "당신은 택배 회사 고객센터 상담원입니다. 고객의 배송 문의에 한국어로 짧고 정확하게 답합니다."},
    {"role": "user", "content": "송장번호를 받았는데 사흘째 배송 조회가 안 됩니다. 무엇을 확인하면 되나요?"},
]

In [ ]:
# [작성] (1) Hub의 GGUF 파일마다 크기와 가중치당 비트를 테이블로 만드세요.
gguf_rows = []
...

gguf_files = pd.DataFrame(gguf_rows)
display(gguf_files)

In [ ]:
# [작성] (2) 4GB GPU에 들어가는 파일 가운데 가중치당 비트가 가장 큰 형식을 고르세요.
office_pick = ...
print("사무실 PC에 올릴 형식:", office_pick)

In [ ]:
# [작성] (3) 교안 02의 measure_tag로 두 태그의 토큰/초와 답을 비교하세요.
tags = ["qwen3.5:4b-q8_0", "qwen3.5:4b-q4_K_M"]

tag_rows = []
...

tag_compare = pd.DataFrame(tag_rows)
display(tag_compare)

In [ ]:
# [작성] (4) 가중치당 비트 비율과 토큰/초 비율을 계산하세요.
bits_ratio = ...
speed_ratio = ...
print("가중치당 비트 Q8_0 / Q4_K_M:", round(bits_ratio, 2), "| 토큰/초 q4_K_M / q8_0:", round(speed_ratio, 2))

In [ ]:
# [자가채점] GGUF 테이블과 고른 형식, 태그 비교 테이블과 두 비율을 확인합니다.
assert isinstance(gguf_files, pd.DataFrame) and len(gguf_files) == 3, "gguf_files는 언어 모델 GGUF 파일 세 개의 3행 DataFrame입니다. .gguf로 끝나고 mmproj로 시작하지 않는 파일만 골랐는지 확인하세요."
assert list(gguf_files.columns) == ["양자화 수준", "크기(바이트)", "크기(GiB)", "가중치당 비트"], "열 이름과 순서를 지문과 같게 쓰세요."
expected_bits = {"Q4_K_M": 5.15, "Q6_K": 6.59, "Q8_0": 8.53}
assert set(gguf_files["양자화 수준"]) == set(expected_bits), "양자화 수준은 파일 이름에서 'Qwen3.5-4B-'와 '.gguf'를 뗀 문자열입니다."
for _, row in gguf_files.iterrows():
    want = expected_bits[row["양자화 수준"]]
    assert abs(row["가중치당 비트"] - want) < 0.01, f"{row['양자화 수준']}의 가중치당 비트가 {row['가중치당 비트']:.3f}입니다. 파일 크기(바이트)에 8을 곱해 파라미터 수로 나눴는지 확인하세요."
assert isinstance(office_pick, str) and office_pick == "Q6_K", "office_pick은 크기(GiB) * 1.2가 4 이하인 파일 가운데 가중치당 비트가 가장 큰 형식입니다. 20% 여유를 더했는지 확인하세요."

assert isinstance(tag_compare, pd.DataFrame) and list(tag_compare.columns) == ["태그", "토큰/초 중앙값", "답"], "tag_compare의 열은 태그, 토큰/초 중앙값, 답입니다."
assert list(tag_compare["태그"]) == tags, "tag_compare는 tags 순서대로 태그마다 한 행입니다."
for _, row in tag_compare.iterrows():
    assert row["토큰/초 중앙값"] > 0, f"{row['태그']}의 토큰/초가 0 이하입니다. measure_tag가 돌려준 첫 값을 담았는지 확인하세요."
    assert isinstance(row["답"], str) and len(row["답"].strip()) > 0, f"{row['태그']}의 답이 비어 있습니다. measure_tag가 돌려준 두 번째 값을 담으세요."
assert bits_ratio > 1 and abs(bits_ratio - gguf_files["가중치당 비트"].max() / gguf_files["가중치당 비트"].min()) < 1e-9, "bits_ratio는 Q8_0 가중치당 비트를 Q4_K_M 가중치당 비트로 나눈 값입니다."
assert speed_ratio > 0, "speed_ratio는 q4_K_M 태그 토큰/초를 q8_0 태그 토큰/초로 나눈 값입니다."
print("✅ 5번 통과")

**결과 읽기**
- **Q4_K_M 5.15비트, Q8_0 8.53비트:** 4비트·8비트보다 큰 것은 묶음마다 스케일을 함께 저장하고, 일부 텐서를 더 높은 비트로 남기기 때문입니다. 어느 텐서를 높은 비트로 남기는지는 만든 곳마다 달라, 같은 Q4_K_M이라도 교안 02 6-3 테이블의 대략값(약 4.9)이나 Ollama `qwen3.5:4b-q4_K_M`(약 5.0비트)과 조금 다릅니다(교안 02 6-3).
- **`Q6_K` 선택:** 생성 시 약 3.87 GiB라 4GB에 들어가고, 약 5.01 GiB인 Q8_0은 넘칩니다. 이미지 입력이 필요해지면 `mmproj` 파일(약 0.63 GiB)도 함께 올려야 하므로 이 판정이 달라집니다.
- **커뮤니티 저장소:** 개발사가 아닌 저장소는 원본 모델(`base_model`)과 라이선스를 모델 카드에서 확인하고 씁니다.
- **비트 비율과 속도 비율:** 8비트 파일은 4비트 파일보다 약 1.66배 큽니다. 토큰 하나를 만들 때마다 가중치를 GPU 메모리에서 읽으므로 4비트 태그가 대개 빠르지만, `speed_ratio`가 비트 비율만큼 커지는지는 GPU와 그때 장치가 하는 일에 따라 다릅니다. 두 태그의 답이 같은지도 테이블에서 확인합니다.

## 6. AWQ·GPTQ 설정 비교

**배경**: 본사 GPU 서버로 서빙할 때는 AWQ·GPTQ 모델을 씁니다(교안 02 6-1). Qwen은 Qwen3.5-4B의 AWQ·GPTQ 저장소를 직접 올리지 않아, 교안 02 6-2처럼 모델 트리(Quantizations)에서 커뮤니티 저장소 두 개를 골라 설정 파일만 읽어 비교합니다. 가중치는 내려받지 않습니다.

- **`QuantTrio/Qwen3.5-4B-AWQ`:** AWQ 형식으로 양자화한 저장소입니다. 설정이 `bits`, `group_size`, `zero_point` 키로 바로 들어 있습니다.
- **`RedHatAI/Qwen3.5-4B-quantized.w4a16`:** llm-compressor로 GPTQ 방식을 적용해 만든 저장소입니다(모델 카드 기준). 이름의 W4A16은 가중치 4비트, 활성값 16비트라는 뜻입니다. 설정의 `quant_method`가 `compressed-tensors`이고, 비트·묶음·대칭 값이 `config_groups` 안의 `group_0` → `weights`에 들어 있습니다.

**요구사항**
- 두 저장소의 설정에서 `quantization_config`(dict)를 꺼내, 저장소마다 한 행인 DataFrame **`quant_compare`**(2행)를 만듭니다. 열은 아래 다섯 개입니다.
  - **`저장소`**: 저장소 전체 이름
  - **`quant_method`**: 설정 값 그대로
  - **`bits`**, **`group_size`**: 비트 수와 묶음 크기. `compressed-tensors`이면 `weights` 안의 `num_bits`, `group_size`를 씁니다.
  - **`대칭`**: 대칭 양자화이면 `True`(bool). `compressed-tensors`이면 `weights` 안의 `symmetric`을, AWQ이면 `zero_point`(0점을 따로 저장하는 비대칭 표시)의 반대를 씁니다.

**확인 기준**: 2행, 다섯 열 이름이고, 행마다 방식·비트·묶음·대칭이 두 설정 파일의 값과 같습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 설정은 교안 01 2-3·3-5의 AutoConfig.from_pretrained(저장소 이름)로 읽습니다. 양자화 저장소면 그 결과의 quantization_config에 dict로 들어 있습니다.
- 저장소마다 quant_method를 먼저 보고, if문으로 설정 모양에 맞게 값을 꺼냅니다.

세부구현:
1. compressed-tensors이면 quant_config["config_groups"]["group_0"]["weights"]를 변수에 담아 세 값을 꺼냅니다.
2. 그 밖에는 quant_config에서 bits, group_size를 바로 꺼내고, 대칭은 not zero_point입니다.
```

</details>

In [ ]:
# [작성] 두 저장소의 양자화 설정을 테이블로 비교하세요.
quant_repos = ["QuantTrio/Qwen3.5-4B-AWQ", "RedHatAI/Qwen3.5-4B-quantized.w4a16"]

quant_rows = []
...

quant_compare = pd.DataFrame(quant_rows)
display(quant_compare)

In [ ]:
# [자가채점] 테이블의 모양과 두 설정의 값을 확인합니다.
assert isinstance(quant_compare, pd.DataFrame) and len(quant_compare) == 2, "quant_compare는 저장소 두 개의 2행 DataFrame입니다."
assert list(quant_compare.columns) == ["저장소", "quant_method", "bits", "group_size", "대칭"], "열 이름과 순서를 지문과 같게 쓰세요."
assert list(quant_compare["저장소"]) == quant_repos, "저장소 열은 quant_repos 순서의 전체 이름입니다."
assert list(quant_compare["quant_method"]) == ["awq", "compressed-tensors"], "quant_method는 설정 값을 그대로 담습니다."
assert list(quant_compare["bits"]) == [4, 4] and list(quant_compare["group_size"]) == [128, 128], "bits와 group_size가 설정 파일과 다릅니다. compressed-tensors는 config_groups 안의 weights에서 num_bits, group_size를 꺼내세요."
assert quant_compare["대칭"].dtype == bool, "대칭 열은 True/False(bool)입니다. 키가 없어 None이 담기지 않았는지 확인하세요."
assert [bool(v) for v in quant_compare["대칭"]] == [False, True], "AWQ는 zero_point: True라 대칭이 아니고(False), RedHatAI 저장소는 weights의 symmetric: True라 대칭(True)입니다."
print("✅ 6번 통과")

**결과 읽기**
- **둘 다 4비트·128개 묶음:** AWQ 저장소는 0점(0이 놓일 정수 위치)도 묶음마다 저장하는 비대칭, RedHatAI 저장소는 교안 02 3-1과 같은 대칭 양자화입니다.
- **설정 모양:** 같은 4비트라도 만든 도구에 따라 설정 파일의 모양이 다릅니다. 저장소 이름보다 `quant_method`와 설정 안의 값을 보고 판단합니다.
- **파일 크기:** Hub의 safetensors 파일 합계는 원본 약 8.68 GiB(이미지 처리 부분 포함), AWQ 약 5.66 GiB, W4A16 약 5.14 GiB입니다. 두 저장소 모두 Gated DeltaNet 층(`linear_attn`)과 이미지 처리 부분, 임베딩을 BF16으로 남기고, AWQ 저장소는 표준 어텐션 층과 첫 층까지 양자화하지 않아(설정의 `modules_to_not_convert`) 더 큽니다.
- **고르기:** 두 저장소 모두 24GB GPU에 여유 있게 들어갑니다. 비트가 적을수록 원본과 답이 더 달라질 수 있으므로 같은 배송 문의 질문 묶음으로 원본과 답을 비교해 고릅니다.

## 판단 쓰기

택배 회사의 GPU가 24GB 한 장이라면 어떤 크기와 형식을 고르고, 배포 전에 무엇을 확인할지 1~6번 결과를 근거로 두세 문장으로 씁니다.

**판단** (두세 문장)

- ...